In [ ]:
import pandas as pd
import MaxNLP
from pathlib import Path
from tqdm.auto import tqdm
import json
tqdm.pandas(desc="Applying")
%load_ext autoreload
%autoreload 2
    
from datetime import date
date_str = date.today().isoformat()

In [ ]:
# Load the translated text sample (25 per country --> 250 samples)
df_t=pd.read_json("2026-05-02 new translated.json")

# load the full data
#df=pd.read_json("clean_df.json")

# Combine responses to one full response
#df["full_response"] = df.apply(MaxNLP.make_one_text, axis=1)

In [ ]:
# Read the codebooks as Markdown

codebook_causes = Path("codebooks/2026-05-06 causes codebook.md").read_text(encoding="utf-8")
codebook_responses = Path("codebooks/2026-01-30 responses codebook.md").read_text(encoding="utf-8")
codebook_consequences = Path("codebooks/2026-04-08b consequences codebook.md").read_text(encoding="utf-8")
codebook_futures = Path("codebooks/2026-02-15 lessons codebook.md").read_text(encoding="utf-8")

# Convert Markdown to Json
codebook_causes_json=MaxNLP.markdown_to_json(codebook_causes)
codebook_responses_json = MaxNLP.markdown_to_json(codebook_responses)
codebook_consequences_json=MaxNLP.markdown_to_json(codebook_consequences)
codebook_futures_json=MaxNLP.markdown_to_json(codebook_futures)


# Get the code list for creating the json scheme.
codes_causes= [i["label"] for i in codebook_causes_json]
codes_responses = [i["label"] for i in codebook_responses_json]
codes_consequences = [i["label"] for i in codebook_consequences_json]
codes_futures = [i["label"] for i in codebook_futures_json]

print(len(codebook_causes.split()))

# Check if all codes are imported with criteria & examples
for i in codebook_causes_json:
    print(i["label"])
    print(len(i['description'].split()),len(i['inclusion_criteria']),len(i['exclusion_criteria']),len(i['included_examples']))

In [ ]:
# Create the System prompt --> requires to create a new client when changing the codebook.

PROMPTBOOK_INSTRUCTIONS = """
You are helpful qualitative research assistant coding short citizen perceptions of the European energy crisis with a defined codebook.
For each label, the codebook includes a definition with inclusion/exclusion criteria and examples that are coded with this label. 
You carefully read the full citizen perception and apply the codebook: What does the citizen say?
You only respond in correct json format. Task: For EACH label, output a confidence score (0.0 = impossible, 1.0 = certain). 
Multi-label allowed. Return JSON only, matching the schema exactly (no extra keys). 
Do not add any labels that are not in the schema.
This is your codebook for the question "What caused the EU energy crisis?":\n
""".strip()

In [ ]:
# Load the Nebula Client
from dotenv import load_dotenv
import openAI_key

client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
available_models = MaxNLP.get_nebula_models(client)

print(f"Models: {available_models}\n\n")

run_x=1

## Important: Context window is set to 2000 = roughly 2000 words; larger codebooks require to change this.

In [ ]:
# Run the LLM coding in a for-loop

codebook_name="causes"
code_list=codes_causes
codebook=json.dumps(codebook_causes_json, indent=2, ensure_ascii=False) # create a string from the json codebook
text_column='QID8'

import pandas as pd
import jsonlines
from pathlib import Path

for run_x in range(5):

    # Split the set in batches (to rerun not all of them when there is an error)
    tmp = df_t.reset_index(names="orig_index") ### df_t is the translated sample; df is the full sample.
    tmp["batch_row"] = range(len(tmp))
    
    # Send the API request to the LLM; format everything with my module.
    # Write each result as new line to the temp  batch files
    res = tmp.progress_apply(
        lambda r: MaxNLP.code_text(
            r[text_column], r.orig_index, r.batch_row,
            client=client,
            model="FAST.gpt-oss:120b",
            code_list=code_list,
            PROMPTBOOK_INSTRUCTIONS=PROMPTBOOK_INSTRUCTIONS+codebook,
            temperature=0.0,
            force_json_object=False,
            run_x=run_x,
            codebook_name=codebook_name
        ),
        axis=1,
    )

    # Combine the batch files to one json file
    ann_df = pd.DataFrame(x for f in Path("annotations_tmp").glob(f"{date_str} run_{run_x}_{codebook_name}_batch_*.jsonl") for x in jsonlines.open(f)).pipe(
        lambda d: pd.json_normalize(d["annotation"])
        .set_index(d["orig_index"]))
    
    # Writes the full run to one json file
    ann_df.reset_index().to_json(f"{date_str} {codebook_name}_run_{run_x}.json", orient="records")
